### Install Required Packages

In [ ]:
!pip install langchain_openai -q
!pip install langchain_community -q
!pip install langchain -q
!pip install python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Create SQLite Database and Tables

In [2]:
import sqlite3

In [3]:
connection = sqlite3.connect("school.db")

In [4]:
cursor = connection.cursor()

In [5]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS teachers (
    teacher_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    age INTEGER NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS subjects (
    subject_id INTEGER PRIMARY KEY AUTOINCREMENT,
    subject_name TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS teaches (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    teacher_id INTEGER,
    subject_id INTEGER,
    FOREIGN KEY (teacher_id) REFERENCES teachers(teacher_id),
    FOREIGN KEY (subject_id) REFERENCES subjects(subject_id)
)
""")

connection.commit()

### Insert Sample Data into Tables

In [6]:
teachers = [
    ("Kamal", 42),
    ("Saman", 29),
    ("Pawan", 34)
]
cursor.executemany("INSERT INTO teachers (name, age) VALUES (?, ?)", teachers)

subjects = [
    ("Mathematics",),
    ("Science",),
    ("History",),
    ("English",)
]
cursor.executemany("INSERT INTO subjects (subject_name) VALUES (?)", subjects)

teaches = [
    (1, 1),  # Kamal teaches Mathematics
    (1, 2),  # Kamal teaches Science
    (2, 3),  # Saman teaches History
    (3, 4),  # Pawan teaches English
]
cursor.executemany("INSERT INTO teaches (teacher_id, subject_id) VALUES (?, ?)", teaches)

connection.commit()

In [7]:
cursor.execute("SELECT * FROM subjects")
teachers = cursor.fetchall()
print(teachers)

[(1, 'Mathematics'), (2, 'Science'), (3, 'History'), (4, 'English'), (5, 'Mathematics'), (6, 'Science'), (7, 'History'), (8, 'English')]


In [8]:
cursor.execute("""SELECT t.name
FROM teachers t
JOIN teaches te ON t.teacher_id = te.teacher_id
JOIN subjects s ON te.subject_id = s.subject_id
WHERE s.subject_name = 'Mathematics';""")
teachers = cursor.fetchall()
print(teachers)

[('Kamal',), ('Kamal',)]


### Initialize LangChain SQL Database

In [10]:
from dotenv import load_dotenv
import os

# Load .env file
load_dotenv()

# Get API key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("API Key Loaded !!!")

API Key Loaded !!!


In [11]:
from langchain_community.utilities.sql_database import SQLDatabase

C:\Users\damsara\AppData\Local\Temp\ipykernel_26152\2983476473.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities.sql_database import SQLDatabase


In [12]:
db = SQLDatabase.from_uri(f"sqlite:///school.db")

In [13]:
print(db.get_usable_table_names())

['subjects', 'teachers', 'teaches']


In [14]:
print(db.table_info)


CREATE TABLE subjects (
	subject_id INTEGER, 
	subject_name TEXT NOT NULL, 
	PRIMARY KEY (subject_id)
)

/*
3 rows from subjects table:
subject_id	subject_name
1	Mathematics
2	Science
3	History
*/


CREATE TABLE teachers (
	teacher_id INTEGER, 
	name TEXT NOT NULL, 
	age INTEGER NOT NULL, 
	PRIMARY KEY (teacher_id)
)

/*
3 rows from teachers table:
teacher_id	name	age
1	Kamal	42
2	Saman	29
3	Pawan	34
*/


CREATE TABLE teaches (
	id INTEGER, 
	teacher_id INTEGER, 
	subject_id INTEGER, 
	PRIMARY KEY (id), 
	FOREIGN KEY(teacher_id) REFERENCES teachers (teacher_id), 
	FOREIGN KEY(subject_id) REFERENCES subjects (subject_id)
)

/*
3 rows from teaches table:
id	teacher_id	subject_id
1	1	1
2	1	2
3	2	3
*/


### Initialize LLM

In [15]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY")
)


### Create SQL Generation Chain

In [16]:
from langchain_classic.chains.sql_database.query import create_sql_query_chain

query_generate = create_sql_query_chain(llm, db)

### Setup SQL Execution Tool

In [17]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

# This is what was missing — needed for Cell 16
query_execute = QuerySQLDataBaseTool(db=db)

C:\Users\damsara\AppData\Local\Temp\ipykernel_26152\3842575556.py:9: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the `langchain-community package and should be used instead. To use it run `pip install -U `langchain-community` and import as `from `langchain_community.tools import QuerySQLDatabaseTool``.
  query_execute = QuerySQLDataBaseTool(db=db)


In [24]:
import re

def extract_sql(raw_query: str) -> str:
    """Extract clean SQL from LLM output that may include Question:/SQLQuery: prefixes."""
    text = raw_query.strip()
    
    # If the model echoed "SQLQuery:" before the actual SQL, take what's after it
    if "SQLQuery:" in text:
        text = text.split("SQLQuery:")[-1].strip()
    
    # Remove anything after "SQLResult:" or "Answer:" if present
    for stop_word in ["SQLResult:", "Answer:"]:
        if stop_word in text:
            text = text.split(stop_word)[0].strip()
    
    # Strip markdown code fences if present
    text = re.sub(r"^```sql\s*|```$", "", text, flags=re.MULTILINE).strip()
    
    return text

raw_query = query_generate.invoke({"question": "Which teachers are assigned to teach English?"})
query = extract_sql(raw_query)
print(query)
print("################################################")
result = query_execute.invoke(query)
print(result)

SELECT T."name" FROM teachers T INNER JOIN teaches TC ON T."teacher_id" = TC."teacher_id" INNER JOIN subjects S ON TC."subject_id" = S."subject_id" WHERE S."subject_name" = 'English' LIMIT 5
################################################
[('Pawan',), ('Pawan',)]


### Create Answer Generator Chain

In [29]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

answer_prompt = PromptTemplate.from_template(
    """Given a user question, the generated SQL query, and its result, write a clear and natural answer to the question.

    User Question: {question}
    SQL Query: {query}
    SQL Result: {result}

    Final Answer:"""
)

final_answer = answer_prompt | llm | StrOutputParser()

In [30]:
from langchain_core.runnables import RunnableLambda

# Reuses the extract_sql() function defined in Cell 15
chain = (
    RunnablePassthrough.assign(
        query=query_generate | RunnableLambda(extract_sql)
    ).assign(
        result=itemgetter("query") | query_execute
    )
    | final_answer
)

In [31]:
chain.invoke({"question": "Details of all teachers"})

'Here are the details of the first 5 teachers: \n\n1. Teacher ID: 1, Name: Kamal, Age: 42\n2. Teacher ID: 2, Name: Saman, Age: 29\n3. Teacher ID: 3, Name: Pawan, Age: 34\n4. Teacher ID: 4, Name: Kamal, Age: 42\n5. Teacher ID: 5, Name: Saman, Age: 29\n\nNote that there are duplicate names and ages in the list, suggesting that there may be multiple teachers with the same name and age.'

In [32]:
chain.invoke({"question": "Which teachers are assigned to teach Mathematics?"})

'The teacher assigned to teach Mathematics is Kamal. Note that Kamal is listed twice in the result, but this likely indicates that Kamal is teaching multiple sections of Mathematics rather than being two separate teachers.'